# RAG Semantic Match (vLLM)

For each generated symptom string, find the closest term in the
Symptom Ontology using a two-stage pipeline:

1. **Dense retrieval** — embed the symptom with `intfloat/e5-large`
   and FAISS-search the ontology for the top-k nearest labels.
2. **LLM rerank** — ask a locally hosted vLLM model to pick the
   single best ontology label from those candidates (or `NONE`).

If the LLM refuses or fails to parse, fall back to the embedding
top-1. Final result includes the canonical SYMP id when one is
found.


## 0. (Optional) Install dependencies

In [ ]:
# Uncomment on a fresh environment:
# !pip install sentence-transformers faiss-cpu


## 1. Optional: tee notebook output to a log file

Useful for long, unattended runs over SSH. Disabled by default
because permanently redirecting `sys.stdout` makes the notebook
look hung in the browser.


In [ ]:
# import sys
# _orig_stdout = sys.stdout
# sys.stdout = open("pipeline.log", "a")
#
# To watch progress from a terminal while this runs:
#     tail -f pipeline.log
#
# To restore normal cell output afterwards:
#     sys.stdout = _orig_stdout


## 2. Pick the vLLM model to query

In [ ]:
# Other models we have benchmarked against the same prompt:
# vllm_model = "Qwen/Qwen2.5-7B-Instruct"
# vllm_model = "Qwen/Qwen2.5-14B-Instruct"
# vllm_model = "mistralai/Mistral-7B-Instruct-v0.2"
# vllm_model = "microsoft/Phi-3-mini-4k-instruct"
# vllm_model = "google/gemma-3-4b-it"
# vllm_model = "meta-llama/Llama-3.1-8B"

vllm_model = "google/gemma-7b-it"


## 3. Build the FAISS index over ontology labels

We normalize embeddings and use `IndexFlatIP`, which makes inner
product equivalent to cosine similarity.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss


def build_ontology_index(labels):
    """Return (embed_model, faiss_index, label_embeddings)."""
    embed_model = SentenceTransformer("intfloat/e5-large", device="cpu")
    embeddings = embed_model.encode(labels, normalize_embeddings=True)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine sim because vectors are normalized
    index.add(embeddings)

    return embed_model, index, embeddings


def retrieve_candidates(symptom, embed_model, index, labels, k=30):
    """Top-k ontology labels nearest to `symptom` in embedding space."""
    query = embed_model.encode([symptom], normalize_embeddings=True)
    scores, idx = index.search(query, k)
    return [labels[i] for i in idx[0]]


## 4. LLM reranker

The prompt forces a `<json>...</json>` envelope so we can extract
the answer with a regex even when the model adds chatter around it.


In [ ]:
import requests
import re


def best_match_rag(model, symptom, candidates):
    """Ask the LLM to pick the single best candidate label (or NONE)."""
    prompt = f"""
Match the symptom to the closest ontology label.

Symptom: "{symptom}"

Candidate labels:
{json.dumps(candidates)}

Return ONLY:

<json>
{{
  "symptom": "{symptom}",
  "best_match": "<exact label from the list or NONE>"
}}
</json>
"""

    r = requests.post(
        "http://127.0.0.1:8000/v1/chat/completions",
        json={
            "model": model,
            "messages": [
                {"role": "user", "content": prompt},
            ],
            "max_tokens": 128,
            "temperature": 0.0,  # deterministic: we want the same pick every call
        },
        timeout=60,
    )

    data = r.json()
    try:
        text = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError):
        return None

    # Pull the JSON envelope out, case-insensitive and across newlines.
    match = re.search(r"(?is)<json>\s*(.*?)\s*</json>", text.strip())
    if not match:
        return None

    try:
        return json.loads(match.group(1).strip())
    except json.JSONDecodeError:
        return None


## 5. Ontology loaders + JSONL reader

The ontology file may be stored either as dict-per-line
(`{"id": ..., "label": ...}`) or as a 2-tuple per line — we accept
both.


In [ ]:
import json


def load_onto_symp_labels(path):
    """Return just the human-readable labels."""
    labels = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict):
                labels.append(obj["label"])
            elif isinstance(obj, (list, tuple)) and len(obj) == 2:
                labels.append(obj[1])
            else:
                raise ValueError(f"Unrecognized ontology line format: {obj}")
    return labels


def load_onto_symp_terms(path):
    """Return (id, label) pairs."""
    terms = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict):
                terms.append((obj["id"], obj["label"]))
            elif isinstance(obj, (list, tuple)) and len(obj) == 2:
                terms.append((obj[0], obj[1]))
            else:
                raise ValueError(f"Unrecognized ontology line format: {obj}")
    return terms


def read_jsonl(path):
    """Yield each JSON line; skip and report malformed lines."""
    with open(path) as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Skipping bad line {i}: {e}")
                continue


## 6. Main pipeline: associations → semantic matches

For every generated `(Disease, Symptom)` record:

1. retrieve top-30 ontology candidates by embedding similarity,
2. ask the LLM to pick one,
3. on LLM failure, fall back to the top-1 embedding hit,
4. look up the canonical SYMP id for the chosen label,
5. yield a status-tagged record.


In [ ]:
def process_associations(
    assoc_path="../outs/associations-generated-01102026.jsonl",
    onto_symp_path="../data/onto-symp-labels.jsonl",
    model=vllm_model,
):
    labels = load_onto_symp_labels(onto_symp_path)
    terms = load_onto_symp_terms(onto_symp_path)

    # Lowercase index so LLM-returned label casing doesn't matter.
    id_by_label = {label.lower(): term_id for term_id, label in terms}

    embed_model, index, _ = build_ontology_index(labels)

    for assoc in read_jsonl(assoc_path):
        try:
            symptom = assoc["Symptom"]
            candidates = retrieve_candidates(symptom, embed_model, index, labels)

            # 1. Single LLM attempt (no retries).
            match = best_match_rag(model, symptom, candidates)
            if not match or not isinstance(match, dict):
                best_label = None
            else:
                best_label = match.get("best_match")

            # 2. Fallback to top-1 embedding hit if LLM refused.
            if not best_label or best_label.upper() == "NONE":
                best_label = candidates[0] if candidates else None

            # 3. If even embeddings produced nothing, emit a clean failure.
            if not best_label:
                yield {
                    "generated_symptom": symptom,
                    "onto_best_match": None,
                    "onto_term_id": None,
                    "status": "SEMANTIC MATCH WITHOUT ID",
                }
                continue

            # 4. Normalize then look up the SYMP id.
            best_label_clean = best_label.strip().lower()
            onto_id = id_by_label.get(best_label_clean)

            # 5. Yield final, status-tagged record.
            yield {
                "generated_symptom": symptom,
                "onto_best_match": best_label,
                "onto_term_id": onto_id,
                "status": "SEMANTIC MATCH WITH ID" if onto_id else "SEMANTIC MATCH WITHOUT ID",
            }

        except Exception as e:
            # Never let a single bad record kill the whole run.
            yield {
                "generated_symptom": assoc.get("Symptom"),
                "onto_best_match": None,
                "onto_term_id": None,
                "status": f"ERROR: {e}",
            }
            continue


## 7. Drive the pipeline and collect results

In [ ]:
results = []
counter = 0

for result in process_associations():
    # Heartbeat so we can see the run is alive on long jobs.
    if counter % 100 == 0:
        print("Current counter:", counter)
        print(result)
    results.append(result)
    counter += 1


Current counter: 0
{'generated_symptom': 'muscle cramps', 'onto_best_match': 'muscle cramp', 'onto_term_id': 'SYMP:0000093', 'status': 'SEMANTIC MATCH WITH ID'}
Current counter: 100
{'generated_symptom': 'myxedema', 'onto_best_match': 'edema', 'onto_term_id': 'SYMP:0000538', 'status': 'SEMANTIC MATCH WITH ID'}
Current counter: 200
{'generated_symptom': 'photosensitivity', 'onto_best_match': 'light sensitivity', 'onto_term_id': 'SYMP:0019165', 'status': 'SEMANTIC MATCH WITH ID'}
Current counter: 300
{'generated_symptom': 'arthritis', 'onto_best_match': 'arthritis', 'onto_term_id': 'SYMP:0019169', 'status': 'SEMANTIC MATCH WITH ID'}
Current counter: 400
{'generated_symptom': 'abdominal mass', 'onto_best_match': 'abdominal mass', 'onto_term_id': 'SYMP:0000798', 'status': 'SEMANTIC MATCH WITH ID'}
Current counter: 500
{'generated_symptom': 'photosensitivity', 'onto_best_match': 'light sensitivity', 'onto_term_id': 'SYMP:0019165', 'status': 'SEMANTIC MATCH WITH ID'}
Current counter: 600
{'g

## 8. Persist results to disk

In [ ]:
# Filename encodes which vLLM model produced these matches.
with open("../outs/semsymp-analysis-gemma-7b-it.jsonl", "w") as f:
    for item in results:
        f.write(json.dumps(item) + "\n")


In [ ]:
print("The job is done and the file should be a permanent resident in the [outs] directory :-D)")
len(results)


## 9. (Debug) — current Python process id

In [ ]:
import os

os.getpid()
